In [ ]:
import argparse
from typing import List
from utils.metrics1 import cacluate_interval_score,evaluate_regress
from utils.tools import converge
from data.data_process import DataLoader
from models.custom_lgbm import CustomLightgbm
from models.custom_hgbt import CustomHGBoost
from models.custom_catb import CustomCatBoost
from models.custom_tabpfn import CustomTabPFN
from models.custom_ngbt import CustomNGBoost
from models.custom_baye import CustomBaye
from models.custom_tabm1 import CustomTABM1,TabMConfig
from models.custom_lstmr import CustomLSTM1
from models.custom_mlp import CustomMLP
from models.custom_tabilc import CustomTabICL
from pathlib import Path
import numpy as np
import pandas as pd
import gc
import os
from utils.tools import set_seed
import torch
import matplotlib.pyplot as plt

set_seed(3407)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
  
day_set=100   # train day set inclunding(train_valid_test)

zhixin=np.array([0,0.3,0.5,0.7,0.9]) # prediction confidence levelsnfidential 
# the first value must be 0, corresponding to the 0.5 quantile (deterministic point prediction),
# while the remaining confidence levels can be flexibly specified in ascending order,
#  e.g., [0, 0.3, 0.9] or [0, 0.1, 0.2, 0.3, 0.6, 0.9]

zhixin1=zhixin[1:]

data_set_list = ['data_JS', 'data_FBS', 'data_NG', 'data_AU', 'data_GED']
# data_set_list = [ 'data_JS', 'data_NG', 'data_AU']
# Geographic information for different datasets
site_config = {
    'data_FBS': {
        'std_meridian_deg': 120,
        'longitude': 104.0,
        'latitude': 37.0
    },
    'data_JS': {
        'std_meridian_deg': 120,
        'longitude': 93.4,
        'latitude': 36.0
    },
    'data_NG': {
        'std_meridian_deg': 120,
        'longitude': 113.4,
        'latitude': 38.0
    },
    'data_GED': {
        'std_meridian_deg': 0,
        'longitude': 133.87,
        'latitude': -23.7
    },
    'data_AU': {
        'std_meridian_deg': 120,
        'longitude': 130.98,
        'latitude': -25.24
    }
}

augment_index=0
augment_index_label='_noagu'
if augment_index==1:
    augment_index_label='_agument'
# print('Dataset:', data_test_name)
# print('Standard meridian:', std_meridian_deg)
# print('Longitude:', longitude)
# print('Latitude:', latitude)

method_choose_list=['Baye','LightGBM','TabICL','TabPFN','TABM','NGBoost','MLP','LSTM','KAN']
# method_choose_list=['TabICL','LightGBM']

method_choose_list1=[]
prdict_y_list_all=[]
true_y_list_all=[]
lower_y_list_all=[]
up_y_list_all=[]
output_df_ngboost_all=[]
point_predict_all=[]
interval_predict_all=[]
df_test_list_all=[]

index=np.arange(0,len(method_choose_list))  #method choose
# index=[0,1,2,3,4,5,6]

# index=[1,4]
# index=[4,5,6,7,8,9,10]
# index=[0,1,2,3,4,5,9]
# index=[7,8,9,10]
# index=[1,2,4]

for index_set in index:
    method_choose=method_choose_list[index_set]
    method_choose_list1.append(method_choose)
    station_name_list=[] #record the station_name of all eara
    pv_station_data_list=[] #record the data of all station
    point_predict=[]  
    interval_predict=[]  
    true_y_list=[]
    prdict_y_list=[]
    lower_y_list=[]
    up_y_list=[]
    model_list=[]
    standard_list=[]
    mae_vaild=0

    for data_test_name in data_set_list:
        project_root = Path.cwd()
        dataset_folder_path = project_root / 'datasets' / data_test_name

        print(dataset_folder_path)
        
        dataset_paths = os.listdir(dataset_folder_path)
        std_meridian_deg = site_config[data_test_name]['std_meridian_deg']
        longitude = site_config[data_test_name]['longitude']
        latitude = site_config[data_test_name]['latitude']

        for i in range(0,len(dataset_paths)):
        # for i in range(0,1):
            fafian_num=i

            dataset_path = dataset_paths[i]
            # print(dataset_path) 

            station_name = dataset_path[:-5] 
            # print(station_name) 
            station_name_list.append(station_name)

            df_all = pd.read_excel(dataset_folder_path / dataset_path)

            df_process=df_all.dropna()

            df_process['row_index'] =np.arange(len(df_process))

            df_augmented = df_process.copy()

            method_choose1=method_choose
        
            df_train_list=df_process[df_process['row_index']<=day_set*0.8*96]

            df_vaild_list=df_process[(df_process['row_index']> day_set*0.8*96) & (df_process['row_index']<=0.9*day_set*96)]
            
            df_test_list=df_process[(df_process['row_index']> day_set*0.9*96) & (df_process['row_index']<=day_set*96)]

            drop_column=['row_index']

            df_train_list1=df_train_list.copy()
            df_train_list1=df_train_list1.drop(drop_column , axis=1)

            df_vaild_list1=df_vaild_list.copy()
            df_vaild_list1=df_vaild_list1.drop(drop_column , axis=1)

            df_test_list1=df_test_list.copy()
            df_test_list1=df_test_list1.drop(drop_column , axis=1)
            
            dataloader = DataLoader(latitude,longitude,std_meridian_deg,df_train_list1,df_vaild_list1,df_test_list1)

            (train_x, train_y, train_x_nor,train_y_nor,
                    train_aug_x, train_aug_y,train_aug_x_nor,train_aug_y_nor,
                    val_x, val_y,val_x_nor, val_y_nor,
                    val_aug_x, val_aug_y,val_aug_x_nor, val_aug_y_nor,
                    test_x, test_y,test_x_nor, test_y_nor,
                    test_aug_x, test_aug_y, test_aug_x_nor, test_aug_y_nor,
                    y_mean, y_std) = dataloader.get_dataset()
            
            if augment_index!=1:
                train_aug_x_nor=train_x_nor
                val_aug_x_nor=val_x_nor
                test_aug_x_nor=test_x_nor
            
            if method_choose1=='Baye':
                beya_model = CustomBaye(zhixin)
                beya_model.train( train_aug_x_nor, train_y_nor,val_aug_x_nor, val_y_nor)
                y_middle, upper, lower=beya_model.predict(test_aug_x_nor, y_mean, y_std)

            elif method_choose1=='LightGBM':
                lgbm_model = CustomLightgbm(zhixin)
                lgbm_model.train( train_aug_x_nor, train_y_nor,val_aug_x_nor, val_y_nor)
                y_middle, upper, lower=lgbm_model.predict(test_aug_x_nor, y_mean, y_std)

            elif method_choose1=='TabPFN':
                tabpfn_au_model = CustomTabPFN(zhixin)
                tabpfn_au_model.train( train_aug_x_nor, train_y_nor,val_aug_x_nor, val_y_nor )
                y_middle, upper, lower=tabpfn_au_model.predict(test_aug_x_nor, y_mean, y_std)

            elif method_choose1=='TabICL':
                tabicl_model = CustomTabICL(zhixin)
                tabicl_model.train( train_aug_x_nor, train_y_nor,val_aug_x_nor, val_y_nor )
                y_middle, upper, lower=tabicl_model.predict(test_aug_x_nor, y_mean, y_std)

            elif method_choose1=='CatBoost':
                catboost_model = CustomCatBoost(zhixin)
                catboost_model.train( train_aug_x_nor, train_y_nor,val_aug_x_nor, val_y_nor)
                y_middle, upper, lower=catboost_model.predict(test_aug_x_nor, y_mean, y_std)

            elif method_choose1=='TABM':
                
                gc.collect()
                torch.cuda.empty_cache()
                
                config = TabMConfig(
                task_type='regression',
                n_num_features=train_aug_x_nor.shape[1],
                cat_cardinalities=None,
                n_classes=None,
                num_embedding_type='none',   # 'none'|'linear_relu'|'periodic'|'piecewise'
                # regression_label_stats=regression_label_stats,
                lr=5e-3, weight_decay=3e-4,
                batch_size=256, num_epochs=50,
                gradient_clipping_norm=1.0,
                enable_amp=False,                # 需要再改 True
                compile_model=False)
                model = CustomTABM1(zhixin,config)  
                model.train(train_aug_x_nor, train_y_nor,val_aug_x_nor, val_y_nor)
                torch.cuda.empty_cache()
                y_middle, upper, lower=model.predict(test_aug_x_nor,y_mean,y_std)  

            elif method_choose1=='LSTM':
                gc.collect()
                torch.cuda.empty_cache()
                lstm_model = CustomLSTM1(zhixin,hidden_dim=64,num_layers=1,lr=0.001,batch_size=256,num_epochs=60,method_choose='LSTM')
                lstm_model.train( train_aug_x_nor, train_y_nor,val_aug_x_nor, val_y_nor)            
                y_middle, upper, lower=lstm_model.predict(test_aug_x_nor, y_mean, y_std)

            elif method_choose1=='GRU':
                gc.collect()
                torch.cuda.empty_cache()
                gru_model = CustomLSTM1(zhixin,hidden_dim=64,num_layers=1,lr=0.001,batch_size=256,num_epochs=60,method_choose='GRU')
                gru_model.train( train_aug_x_nor, train_y_nor,val_aug_x_nor, val_y_nor)            
                y_middle, upper, lower=gru_model.predict(test_aug_x_nor, y_mean, y_std)

            elif method_choose1=='MLP':
                gc.collect()
                torch.cuda.empty_cache()
                mlp_model = CustomMLP(zhixin,input_dim=train_aug_x_nor.shape[1],hidden_dim=32,lr=0.001,batch_size=256,num_epochs=60,method_choose='MLP')
                mlp_model.train( train_aug_x_nor, train_y_nor,val_aug_x_nor, val_y_nor)                     
                y_middle, upper, lower=mlp_model.predict(test_aug_x_nor, y_mean, y_std)

            elif method_choose1=='KAN':
                gc.collect()
                torch.cuda.empty_cache()
                kan_model = CustomMLP(zhixin,input_dim=train_aug_x_nor.shape[1],hidden_dim=32,lr=0.001,batch_size=256,num_epochs=60,method_choose='KAN')
                kan_model.train( train_aug_x_nor, train_y_nor,val_aug_x_nor, val_y_nor)                     
                y_middle, upper, lower=kan_model.predict(test_aug_x_nor, y_mean, y_std)
    
            elif method_choose1=='NGBoost':
                ngboost_model = CustomNGBoost(zhixin)
                ngboost_model.train( train_aug_x_nor, train_y_nor, val_aug_x_nor,val_y_nor)
                y_middle, upper, lower=ngboost_model.predict(test_aug_x_nor, y_mean, y_std)
                
            elif method_choose1=='HGBoost':
                hgboost_model = CustomHGBoost(zhixin)
                hgboost_model.train( train_aug_x_nor, train_y_nor, val_aug_x_nor,val_y_nor)
                y_middle, upper, lower=hgboost_model.predict(test_aug_x_nor, y_mean, y_std)            

            true_y_list.append(test_y)
            prdict_y_list.append(y_middle)
            lower_y_list.append(lower)
            up_y_list.append(upper)

            MAE,MCE,MSE,RMSE,R2=evaluate_regress(y_middle, test_y)

            point_predict.append([MAE,MCE,MSE,RMSE,R2])

            quantile =1-(1-zhixin1)/2  
            # print(quantile)
            quantile1 =(1-zhixin1)/2  
            cacluate_interval=np.ones((5,len(quantile)))
            
            for i in range(len(quantile)):
                        # confidence_level = 0.95
                        confidence_level=zhixin1[i]
                        alpha=1-confidence_level
                        y_true=test_y
                        lower_bounds =lower[:,i]
                        upper_bounds=upper[:,i]
                        picp_result,pinaw_result,cwc_result,ql_result,interval_score_result=cacluate_interval_score(y_true, lower_bounds, upper_bounds, 10, alpha)

                        cacluate_interval[:,i]=[picp_result,pinaw_result,cwc_result,ql_result,interval_score_result]

            # print(y_mean)
            interval_predict.append(cacluate_interval)
            df_test_list_all.append(df_test_list)

            output_df=converge(station_name_list,df_test_list,true_y_list,prdict_y_list,up_y_list,lower_y_list,zhixin1) #概率区间预测汇集结果        
    
    
    true_y_list_all.append(true_y_list)  
    prdict_y_list_all.append(prdict_y_list)      
    lower_y_list_all.append(lower_y_list)
    up_y_list_all.append(up_y_list)    
    output_df_ngboost_all.append(output_df)
    point_predict_all.append(point_predict)
    interval_predict_all.append(interval_predict)

    MCE_array=np.zeros(len(point_predict))
    for i in range(len(point_predict)):
        MCE_array[i]=point_predict[i][1]
    print('*******************************************')
    print(method_choose+'_NRMSE')
    print(MCE_array)
 
    interval_MCE_array=np.zeros(len(point_predict))
    for i in range(len(interval_predict)):
        for j in range(0,4):
            interval_MCE_array[i]=interval_MCE_array[i]+interval_predict[i][3,j]
    interval_MCE_array1=interval_MCE_array/4

    print('*******************************************')
    print(method_choose+'_IS')
    print(interval_MCE_array1)

# Metric labels
metrics_label_list = ['MAE', 'NRMSE', 'R2', 'MAPE', 'RMSE',
                      'PICP', 'PINAW', 'CWC', 'IS', 'QL']

# -----------------------------
# Aggregate point prediction metrics
# -----------------------------
point_df_list = []

for n in range(5):
    result1 = np.zeros((len(point_predict_all[0]), len(point_predict_all)))

    for i in range(len(point_predict_all)):          # Loop over methods
        for j in range(len(point_predict_all[0])):   # Loop over stations
            result1[j, i] = np.mean(point_predict_all[i][j][n])

    result1_df = pd.DataFrame(
        result1,
        index=station_name_list,
        columns=method_choose_list1
    )
    point_df_list.append(result1_df)

result3_df = pd.concat(point_df_list, axis=1)

# -----------------------------
# Aggregate interval prediction metrics
# -----------------------------
interval_df_list = []

for n in range(5):
    result1 = np.zeros((len(interval_predict_all[0]), len(interval_predict_all)))

    for i in range(len(interval_predict_all)):          # Loop over methods
        for j in range(len(interval_predict_all[0])):   # Loop over stations
            result1[j, i] = np.mean(interval_predict_all[i][j][n, :])

    result1_df = pd.DataFrame(
        result1,
        index=station_name_list,
        columns=method_choose_list1
    )
    interval_df_list.append(result1_df)

result2_df = pd.concat(interval_df_list, axis=1)

# -----------------------------
# Combine point and interval metrics
# -----------------------------
result2_df_last = pd.concat([result3_df, result2_df], axis=1)

# -----------------------------
# Build a two-level column index
# Level 1: metric name
# Level 2: method name
# -----------------------------
multi_cols = pd.MultiIndex.from_tuples(
    [(metric, method) for metric in metrics_label_list for method in method_choose_list1],
    names=['Metric', 'Method']
)

result2_df_last.columns = multi_cols

# Display the final table
print(result2_df_last)

data_test_name1='testday_'+str(day_set)+augment_index_label+'_predict_result1.xlsx'
result2_df_last.to_excel(data_test_name1)


e:\Zwt_pv_test_2023_10_2\SolarUPE_project\datasets\data_JS
e:\Zwt_pv_test_2023_10_2\SolarUPE_project\datasets\data_FBS
e:\Zwt_pv_test_2023_10_2\SolarUPE_project\datasets\data_NG


C:\Users\Administrator\AppData\Local\Temp\ipykernel_16576\2637914236.py:144: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_process['row_index'] =np.arange(len(df_process))


e:\Zwt_pv_test_2023_10_2\SolarUPE_project\datasets\data_AU
e:\Zwt_pv_test_2023_10_2\SolarUPE_project\datasets\data_GED
*******************************************
TabICL_NRMSE
[0.14512232 0.12507797 0.12390191 0.11182672 0.1225221  0.10460838
 0.13656852 0.08020643 0.1052975  0.15132601 0.11853072 0.11828835
 0.10842619 0.10693354 0.11793469 0.13042366 0.11987673 0.13598436
 0.22716334 0.1129207  0.11079151 0.07946886 0.09027797 0.04177792
 0.0432601  0.04203638 0.06144973 0.05315604 0.09651231 0.0434573
 0.05856657 0.112344   0.05867654 0.19909625 0.19750743 0.17950301]
*******************************************
TabICL_IS
[8.16528769e+03 6.35730711e+03 2.68671642e+03 4.64415153e+03
 2.94798777e+03 2.64549446e+03 7.01903717e+03 2.26690488e+03
 7.28364133e+03 9.58719384e+03 2.11676532e+03 1.94134161e+02
 3.72454919e+01 1.42002554e+01 1.24451019e+03 1.38324590e+02
 2.39640011e+03 7.97928066e+00 3.08364425e+03 7.98168260e+01
 5.41928342e+01 7.17816339e+00 2.15413154e+01 6.80157355e+00
 1

C:\Users\Administrator\AppData\Local\Temp\ipykernel_16576\2637914236.py:144: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_process['row_index'] =np.arange(len(df_process))


e:\Zwt_pv_test_2023_10_2\SolarUPE_project\datasets\data_AU
e:\Zwt_pv_test_2023_10_2\SolarUPE_project\datasets\data_GED
*******************************************
LightGBM_NRMSE
[0.14284581 0.0805057  0.09570885 0.11093483 0.09780867 0.10028521
 0.13290678 0.07423583 0.10298611 0.13567571 0.13224836 0.10399075
 0.11001748 0.10659619 0.14604318 0.11578753 0.12931145 0.13023205
 0.18078148 0.11682642 0.10772864 0.07176687 0.1270674  0.03850263
 0.04242742 0.05402474 0.06182197 0.06063506 0.24683425 0.05402569
 0.0598361  0.08110606 0.05971775 0.16999192 0.17503675 0.15097555]
*******************************************
LightGBM_IS
[5.18779603e+03 3.91957297e+03 1.37044299e+03 3.26988476e+03
 1.91878449e+03 2.47444894e+03 6.66309582e+03 2.16399999e+03
 7.35305174e+03 5.97962903e+03 1.70490787e+03 1.76309899e+02
 3.64833532e+01 1.27246116e+01 1.01438537e+03 1.24347275e+02
 1.97623717e+03 7.64157934e+00 1.52648422e+03 7.69761671e+01
 5.12272693e+01 7.35599658e+00 2.38423795e+01 1.02974873e+